# Planar curves: Jacobians, normals, and curvature with dual numbers

Differentiate a one-parameter curve in two-dimensional space using the same
NumPy expression that evaluates its position. `dual1` supplies the first
derivative. Two independent commuting nilpotent directions in `dual2`
supply the second derivative.

The interactive figure links a point on an ellipse to its tangent, normal,
osculating circle arc, and curvature. Run the notebook in order; no
finite-difference step sizes, symbolic differentiation, or widget callbacks
are needed.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.offline import init_notebook_mode
from plotly.subplots import make_subplots
from algpy.algebras import dual1, dual2
from _plotting import slider_controls, vector2

init_notebook_mode(connected=False)
pio.templates.default = "plotly_white"
major, minor = 1.7, 1.0

def curve(t):
    return major*np.cos(t), minor*np.sin(t)

## First order: a 2×1 Jacobian

Since $\varepsilon^2=0$,
$r(t+\varepsilon)=r(t)+r'(t)\varepsilon$.
The two coordinate derivatives form the Jacobian
$J=[dx/dt,dy/dt]^T$. Normalize its column to get the unit tangent, then
rotate that tangent counterclockwise by $90^\circ$ to get the left normal.

In [ ]:
epsilon, = dual1.gens()

def curve_first(t):
    values = curve(dual1(float(t)) + epsilon)
    coefficients = np.array([dual1.to_float(value) for value in values])
    return coefficients[:, 0], coefficients[:, 1:2]

position, jacobian = curve_first(.4)
tangent = jacobian[:, 0]/np.linalg.norm(jacobian)
normal = np.array([-tangent[1], tangent[0]])
np.testing.assert_allclose(jacobian[:, 0], [-major*np.sin(.4), minor*np.cos(.4)], atol=1e-14)
np.testing.assert_allclose(tangent @ normal, 0, atol=1e-14)
print("position:", position)
print("Jacobian (2×1):\n", jacobian)
print("unit tangent:", tangent, "unit normal:", normal)

## Second order: repeated directions need independent generators

Seed $t+\varepsilon_0+\varepsilon_1$, where the two generators commute
and square to zero. The coefficient of $\varepsilon_0\varepsilon_1$ is
$r''(t)$ directly: the factor of two in
$(\varepsilon_0+\varepsilon_1)^2$ cancels the Taylor $2!$.

Each coordinate has a 1×1 Hessian because the curve has one parameter.
Stacking them gives the vector $r''(t)$.

In [ ]:
e0, e1 = dual2.gens()

def curve_second(t):
    values = curve(dual2(float(t)) + e0 + e1)
    coefficients = np.array([dual2.to_float(value) for value in values])
    np.testing.assert_allclose(coefficients[:, 1], coefficients[:, 2], atol=1e-14)
    return coefficients[:, 0], coefficients[:, 1], coefficients[:, 3]

position2, velocity, acceleration = curve_second(.4)
np.testing.assert_allclose(position2, position, atol=1e-14)
np.testing.assert_allclose(velocity, jacobian[:, 0], atol=1e-14)
np.testing.assert_allclose(acceleration, [-major*np.cos(.4), -minor*np.sin(.4)], atol=1e-14)
print("Coordinate Hessians (2×1×1):\n", acceleration[:, None, None])

## Signed curvature and the osculating circle

With speed $v=\|r'\|$, use
\[
\kappa=\frac{x'y''-y'x''}{v^3},\qquad
T=r'/v,\qquad N=(-T_y,T_x).
\]
The osculating circle center is $r+N/\kappa$ and its radius is
$1/|\kappa|$. The counterclockwise ellipse has positive curvature
everywhere, so these quantities stay finite in this example.
The figure draws a local arc of that circle to keep the view readable.

In [ ]:
def curve_state(t):
    point, first, second = curve_second(t)
    speed = np.linalg.norm(first)
    unit_tangent = first/speed
    unit_normal = np.array([-unit_tangent[1], unit_tangent[0]])
    curvature = (first[0]*second[1] - first[1]*second[0])/speed**3
    return point, unit_tangent, unit_normal, curvature, speed

parameters = np.linspace(0, 2*np.pi, 61)
states = [curve_state(t) for t in parameters]
curvatures = np.array([state[3] for state in states])
analytic = major*minor/(major**2*np.sin(parameters)**2 + minor**2*np.cos(parameters)**2)**1.5
np.testing.assert_allclose(curvatures, analytic, rtol=2e-14, atol=1e-14)
np.testing.assert_allclose([np.dot(state[1], state[2]) for state in states], 0, atol=1e-14)
print("All sampled curvatures agree with the analytic ellipse formula.")

In [ ]:
def curve_traces(index):
    point, tangent, normal, curvature, speed = states[index]
    center = point + normal/curvature
    angles = np.linspace(-.55, .55, 45)
    arc = center + (-np.cos(angles)[:, None]*normal + np.sin(angles)[:, None]*tangent)/curvature
    return [
        go.Scatter(x=[point[0]], y=[point[1]], mode="markers",
                   marker=dict(size=11, color="#0f172a"), name="Curve point",
                   text=[f"t={parameters[index]:.3f}<br>speed={speed:.4f}<br>κ={curvature:.4f}"],
                   hovertemplate="%{text}<extra></extra>"),
        vector2(point, tangent, "Unit tangent", "#2563eb"),
        vector2(point, normal, "Left unit normal", "#d97706"),
        go.Scatter(x=arc[:, 0], y=arc[:, 1], mode="lines",
                   line=dict(color="#059669", width=3, dash="dash"), name="Osculating circle arc"),
        go.Scatter(x=[parameters[index]], y=[curvature], mode="markers",
                   marker=dict(size=11, color="#0f172a"), showlegend=False),
    ]

dense_t = np.linspace(0, 2*np.pi, 300)
fig = make_subplots(rows=1, cols=2, column_widths=[.62, .38],
                    subplot_titles=["Position, tangent, and normal", "Signed curvature"])
fig.add_trace(go.Scatter(x=major*np.cos(dense_t), y=minor*np.sin(dense_t),
                        mode="lines", line_color="#94a3b8", name="Ellipse"), row=1, col=1)
initial = curve_traces(0)
for trace in initial[:4]:
    fig.add_trace(trace, row=1, col=1)
fig.add_trace(go.Scatter(x=parameters, y=curvatures, mode="lines",
                        line_color="#db2777", name="κ(t)"), row=1, col=2)
fig.add_trace(initial[4], row=1, col=2)
fig.frames = [go.Frame(name=str(index), data=curve_traces(index), traces=[1, 2, 3, 4, 6])
              for index in range(len(parameters))]
fig.update_xaxes(title_text="x", range=[-2.6, 2.6], row=1, col=1)
fig.update_yaxes(title_text="y", range=[-1.8, 1.8], scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_xaxes(title_text="parameter t", range=[0, 2*np.pi], row=1, col=2)
fig.update_yaxes(title_text="κ", range=[0, 1.9], row=1, col=2)
fig.update_layout(title="Dual-number derivatives reveal the local geometry of an ellipse", height=550)
slider_controls(fig, [f"{t:.2f}" for t in parameters]).show(auto_play=False)

**Try changing** `curve(t)` to another smooth, regular curve, then replace
the ellipse-specific reference check. A zero velocity has no unit tangent;
a zero curvature has no finite osculating circle. The first- and second-order
coefficients themselves are still available wherever the expression is
differentiable.

Continue with `05_surfaces_and_curvature.ipynb` for a 3×2 Jacobian, three
coordinate Hessians, and the two fundamental forms of a surface.